# Staggered DiD in ModernDiD and R

This notebook is generated from the [Staggered DiD in ModernDiD and R](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_staggered.html) page of the ModernDiD documentation.

This comparison checks whether ModernDiD and R's `did` package give the
same results for the minimum wage data that the
[staggered DiD example](https://moderndid.readthedocs.io/en/stable/user_guide/example_staggered_did.html#example-staggered-did) analyzes. We fit
[`att_gt`](https://moderndid.readthedocs.io/en/stable/api/generated/multiperiod/moderndid.att_gt.html) in each language, inspect every cohort-period
estimate, and then compare the summaries from [`aggte`](https://moderndid.readthedocs.io/en/stable/api/generated/multiperiod/moderndid.aggte.html). The
R calls run through `rpy2` on the same data used by Python.

> **What this comparison checks**
>
> Both fits use never-treated counties as controls, a varying base period, and
> log population as a covariate. Both fits use analytical standard errors and
> pointwise confidence intervals. Bootstrap inference and simultaneous bands
> require a separate comparison.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    # Colab ships rpy2 3.5 as a single package. Uninstalling it first keeps pip from deleting
    # the files it shares with the split rpy2 3.6 packages.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "rpy2"], check=True)
    # rpy2 supports Colab's R 4.6 from release 3.6.7 on.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "moderndid[all]==0.3.1", "rpy2>=3.6.7"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "r-cran-did"], check=True)
else:
    print("Outside Colab, set up R as https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running describes.")

## Use the same counties and years

The minimum-wage panel from [`load_mpdta`](https://moderndid.readthedocs.io/en/stable/api/generated/data/moderndid.load_mpdta.html) follows 500 counties
in every year from 2003 through 2007. It records county-level log employment
(`lemp`), log population (`lpop`), and the year a county first received
treatment (`first.treat`, zero for a never-treated county). Treatment starts in
2004 for 20 counties, in 2006 for 40, and in 2007 for 131. The other 309
counties stay untreated through 2007 and form the never-treated comparison
group. Since we pass the same data directly to R, both fits start with the
same observations.

The section on [running the comparisons](https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running)
explains how to install the optional R environment and rerun this notebook.

In [ ]:
import rpy2.robjects as ro

import moderndid as did

data = did.load_mpdta()

In [ ]:
import numpy as np
import pandas as pd
from rpy2.robjects import pandas2ri

with (ro.default_converter + pandas2ri.converter).context():
    ro.globalenv["mpdta"] = ro.conversion.py2rpy(data.to_pandas())

## Fit the group-time effects in Python

We use the doubly robust estimator with the same formula and comparison group
in both packages. Setting `boot=False` and `cband=False` makes the inference
analytical and pointwise.

In [ ]:
py_fit = did.att_gt(
    data=data,
    yname="lemp",
    tname="year",
    idname="countyreal",
    gname="first.treat",
    xformla="~ lpop",
    control_group="nevertreated",
    base_period="varying",
    est_method="dr",
    boot=False,
    cband=False,
)
print(py_fit)

## Fit the same effects in R

The R call names the bootstrap option `bstrap` but otherwise matches the
Python call. The output is R's own summary of its fitted object.

In [ ]:
with ro.default_converter.context():
    ro.r("""
        r_fit <- did::att_gt(
            data = mpdta,
            yname = "lemp",
            tname = "year",
            idname = "countyreal",
            gname = "first.treat",
            xformla = ~ lpop,
            control_group = "nevertreated",
            base_period = "varying",
            est_method = "dr",
            bstrap = FALSE,
            cband = FALSE
        )
    """)
    r_fit = ro.globalenv["r_fit"]
    _ = ro.r("summary(r_fit)")

## Put each cohort-period result next to its counterpart

We join by treatment cohort and calendar year rather than relying on the order
of the returned arrays. The difference columns give Python minus R. Scientific
notation keeps small values visible even when the estimates round to the same
number.

In [ ]:
python_cells = pd.DataFrame({
    "cohort": py_fit.groups,
    "year": py_fit.times,
    "Python ATT": py_fit.att_gt,
    "Python SE": py_fit.se_gt,
})
with ro.default_converter.context():
    r_cells = pd.DataFrame({
        "cohort": r_fit.rx2("group"),
        "year": r_fit.rx2("t"),
        "R ATT": r_fit.rx2("att"),
        "R SE": r_fit.rx2("se"),
    })

comparison = python_cells.merge(r_cells, on=["cohort", "year"], validate="one_to_one")
assert len(comparison) == len(python_cells) == len(r_cells)
comparison["ATT difference"] = comparison["Python ATT"] - comparison["R ATT"]
comparison["SE difference"] = comparison["Python SE"] - comparison["R SE"]
comparison

## Compare the event study

Dynamic aggregation averages across cohorts at each period relative to first
treatment. Negative event times describe each cohort's contrasts before its
own first treatment. The overall dynamic ATT averages the post-treatment
event-time effects.

In [ ]:
py_dynamic = did.aggte(py_fit, type="dynamic", boot=False, cband=False)
print(py_dynamic)

In [ ]:
with ro.default_converter.context():
    ro.r("""
        r_dynamic <- did::aggte(r_fit, type = "dynamic", bstrap = FALSE, cband = FALSE)
    """)
    r_dynamic = ro.globalenv["r_dynamic"]
    _ = ro.r("summary(r_dynamic)")

In [ ]:
python_events = pd.DataFrame({
    "event time": py_dynamic.event_times,
    "Python ATT": py_dynamic.att_by_event,
    "Python SE": py_dynamic.se_by_event,
})
with ro.default_converter.context():
    r_events = pd.DataFrame({
        "event time": r_dynamic.rx2("egt"),
        "R ATT": r_dynamic.rx2("att.egt"),
        "R SE": r_dynamic.rx2("se.egt"),
    })

event_comparison = python_events.merge(r_events, on="event time", validate="one_to_one")
assert len(event_comparison) == len(python_events) == len(r_events)
event_comparison["ATT difference"] = event_comparison["Python ATT"] - event_comparison["R ATT"]
event_comparison["SE difference"] = event_comparison["Python SE"] - event_comparison["R SE"]
event_comparison

## Compare the overall summaries

The aggregation choice changes which effects get averaged and how they are
weighted. Simple aggregation weights all post-treatment cohort-period effects
by cohort size. Group aggregation first averages within each cohort and then
weights the cohort averages. Dynamic aggregation first averages at each event
time. We compare each choice with the same choice in R.

In [ ]:
overall_rows = []
for kind in ("simple", "group", "dynamic"):
    py_aggregate = did.aggte(py_fit, type=kind, boot=False, cband=False)
    with ro.default_converter.context():
        r_aggregate = ro.r(f'did::aggte(r_fit, type = "{kind}", bstrap = FALSE, cband = FALSE)')
        r_att = r_aggregate.rx2("overall.att")[0]
        r_se = r_aggregate.rx2("overall.se")[0]
    overall_rows.append({
        "aggregation": kind,
        "Python ATT": py_aggregate.overall_att,
        "R ATT": r_att,
        "ATT difference": py_aggregate.overall_att - r_att,
        "Python SE": py_aggregate.overall_se,
        "R SE": r_se,
        "SE difference": py_aggregate.overall_se - r_se,
    })
overall_comparison = pd.DataFrame(overall_rows)
overall_comparison

## Check the numerical differences

The tables show the returned values. The notebook also checks every row
against the comparison tolerances and fails on any later run that exceeds them.
Each estimate may differ by at most `1e-6 + 1e-5 * abs(R estimate)`. Each standard error may
differ by at most `1e-6 + 1e-3 * abs(R SE)`. These bounds allow floating-point and
optimization differences without rounding the evidence away.

In [ ]:
for table in (comparison, event_comparison, overall_comparison):
    np.testing.assert_allclose(table["Python ATT"], table["R ATT"], rtol=1e-5, atol=1e-6)
    np.testing.assert_allclose(table["Python SE"], table["R SE"], rtol=1e-3, atol=1e-6)

pd.DataFrame([
    {
        "comparison": label,
        "largest absolute ATT difference": table["ATT difference"].abs().max(),
        "largest absolute SE difference": table["SE difference"].abs().max(),
    }
    for label, table in (
        ("cohort-period effects", comparison),
        ("event-time effects", event_comparison),
        ("overall summaries", overall_comparison),
    )
])

## Clustered inference

With a cluster variable in place of the unit-level standard errors above,
[`att_gt`](https://moderndid.readthedocs.io/en/stable/api/generated/multiperiod/moderndid.att_gt.html) and `did` 2.5.1 both sum the influence functions
within each cluster, as Remark 10 of Callaway and Sant'Anna (2021) describes
for the bootstrap. Given the same draws on clusters of 3 to 46 counties, the
reference tests in `tests/validation/test_r_did.py` find the two packages'
bootstrap standard errors and critical values within 1e-7 of each other in
relative terms. Without the bootstrap, the same tests cluster the minimum wage
data by state and find the analytic cluster-robust standard errors within
1e-8 of each other.

> **Older `did` releases average within clusters**
>
> Releases of `did` before 2.5.0 average the influence function within each
> cluster rather than summing it. When clusters differ in size, their clustered
> bootstrap standard errors therefore depart from Remark 10. Compare ModernDiD
> only with `did` 2.5.0 or later, since those releases sum within clusters as
> the paper does.

Under clustering, the two packages differ only in the pre-test of parallel
trends. With `bstrap = FALSE`, `did` reports a Wald pre-test built from the
clustered covariance (`R/att_gt.R`, lines 559 to 563 and 677 to 679). In
simulations where parallel trends hold and treatment starts at the cluster
level, that test rejected about 20 percent of the time at the 5 percent level
with 50 clusters and six pre-treatment cells. With 200 clusters it still
rejected about 10 percent of the time. Like `did` with the bootstrap,
ModernDiD reports no pre-test for a fit that clusters beyond units and points
to the bootstrap's uniform bands instead.

## Repeated cross-sections and unbalanced panels

With `panel=False` or `allow_unbalanced_panel=True`, both packages estimate each
doubly robust cell with the repeated cross-section estimator of Sant'Anna and
Zhao (2020). `did` 2.5.1 calls the version in `DRDID` 1.3.0 for it
(`R/compute.att_gt2.R`, line 350). The two packages then agree on the estimates
but not, by design, on the standard errors.

Each cell measures its changes from pre-treatment averages that depend on a
regression fitted to the comparison group's pre-treatment outcomes. When that
regression's estimation effect enters the influence function, it takes the
averages' minus sign. `DRDID` adds it with a plus sign instead (`R/drdid_rc.R`,
lines 331 and 332). A finite-difference derivative of the estimate confirms the
minus sign that ModernDiD uses. On the minimum wage data treated as repeated
cross-sections, the cell standard errors differ by at most 0.1 percent, or by up
to 3 percent once they are clustered by state. Before the reference tests in
`tests/validation/test_r_did.py` compare such fits, clustered or not, they
correct that sign inside R.

## Versions used

These versions identify the software that produced the stored output above.
The optional reproduction environment may install newer R reference packages.
Rerunning the notebook records those versions with the new results.

In [ ]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    r_version = ro.r("R.version.string")[0]
    did_version = ro.r('as.character(packageVersion("did"))')[0]
pd.DataFrame({
    "component": ["Python", "ModernDiD", "rpy2", "R", "did"],
    "version": [python_version(), version("moderndid"), version("rpy2"), r_version, did_version],
})